# 単語ベクトルの平均で文ベクトルを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_word_vectors/avg_word_vectors.ipynb)

NLP クックブック 07-07 ／ 単語ベクトル

文や文書をベクトルにする最も手軽な方法として、含まれる単語のベクトルを平均したい。

In [1]:
!pip install -q gensim

In [2]:
import re
import gensim.downloader as api
import numpy as np

wv = api.load("glove-wiki-gigaword-100")

def sent_vec(text):
    words = [w for w in re.findall(r"[a-z]+", text.lower()) if w in wv]
    return np.mean([wv[w] for w in words], axis=0) if words else np.zeros(wv.vector_size)

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "How do I cook pasta?"
docs = ["Boil the spaghetti in salted water for ten minutes.", "The stock market fell sharply today.",
        "A recipe for making noodles at home.", "My cat sleeps all day on the sofa."]
for d in sorted(docs, key=lambda d: -cos(sent_vec(query), sent_vec(d))):
    print(f"{cos(sent_vec(query), sent_vec(d)):.3f}  {d}")

0.788  A recipe for making noodles at home.
0.766  Boil the spaghetti in salted water for ten minutes.
0.757  My cat sleeps all day on the sofa.
0.555  The stock market fell sharply today.


In [3]:
# SIF 風の重み付け: 高頻度語の重みを下げる (順位を頻度の代わりに使った簡易版)
def weighted_vec(text, a=1e-3):
    words = [w for w in re.findall(r"[a-z]+", text.lower()) if w in wv]
    weights = [a / (a + 1 / (wv.key_to_index[w] + 2) / 10) for w in words]  # Zipf 則で順位から頻度を近似
    return np.average([wv[w] for w in words], axis=0, weights=weights)

for d in sorted(docs, key=lambda d: -cos(weighted_vec(query), weighted_vec(d))):
    print(f"{cos(weighted_vec(query), weighted_vec(d)):.3f}  {d}")

0.827  A recipe for making noodles at home.
0.754  Boil the spaghetti in salted water for ten minutes.
0.596  My cat sleeps all day on the sofa.
0.433  The stock market fell sharply today.


- 語順は無視されるが、驚くほど強いベースライン。まずこれを試し、足りなければ文埋め込みモデル (次章) に進む。
- 単純平均は `the` `of` などの機能語に引っ張られて、どの文も似てしまう。ストップワード除去か頻度による重み付けが効く。